Prompt Engineering: https://github.com/NirDiamant/Prompt_Engineering/blob/main/all_prompt_engineering_techniques/basic-prompt-structures.ipynb

In [ ]:
import torch
print(torch.cuda.is_available())       # should print: True
print(torch.cuda.get_device_name(0))   # should print: Tesla T4
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
from transformers import pipeline
login("hf_Pt")

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
def ask(
    prompt,
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True,
    system="You are a helpful and concise AI assistant."
):
    """
    Sends a prompt to the loaded model and returns the response text.

    Parameters:
    -----------
    prompt          : str   — The user's message or instruction
    max_new_tokens  : int   — Maximum length of the response (in tokens, ~¾ of a word each)
    temperature     : float — Creativity level: 0.0 = deterministic, 1.0 = very creative
    do_sample       : bool  — If False, always picks the most likely next token (greedy)
    system          : str   — The system prompt (model's 'role' or 'persona')
    """
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]

    output = pipe(
        messages,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
    )

    # Extract just the assistant's reply
    return output[0]["generated_text"][-1]["content"]


# Quick sanity test
# print(ask("Say hello in exactly 5 words."))

Technique 1 — Introduction: What Is a Prompt?

Concept: A prompt is the full input you give the model. Small wording changes create dramatically different outputs. This first exercise makes that viscerally clear.

What to notice: The raw content (black holes) never changed. Everything else — tone, depth, format — was shaped entirely by how you asked.

In [ ]:
# Same question, three different framings
q = "Tell me about black holes."

v1 = ask(q)
v2 = ask("Explain black holes to a 10-year-old in 3 sentences.")
v3 = ask("You are a dramatic poet. Describe black holes in 4 lines of verse.")

print("=== VERSION 1 (bare question) ===\n", v1)
print("\n=== VERSION 2 (audience + constraint) ===\n", v2)
print("\n=== VERSION 3 (role + format) ===\n", v3)

Technique 2 — Basic Prompt Structures: Single-Turn vs Multi-Turn

Concept: A single-turn prompt is one message → one answer. Multi-turn is a conversation where context builds.


Key lesson: In multi-turn, the model "remembers" what it said earlier. This context window is finite — it's one of the most important practical constraints in prompt engineering.

In [ ]:
# --- Single-turn ---
single = ask("What is photosynthesis?")
print("SINGLE-TURN:\n", single)

# --- Multi-turn (manual history) ---
# We build up the conversation manually
history = [
    {"role": "system",    "content": "You are a friendly science teacher."},
    {"role": "user",      "content": "What is photosynthesis?"},
    {"role": "assistant", "content": single},   # reuse the answer from above
    {"role": "user",      "content": "Can plants do it at night?"},
]

# This line uses the 'pipe' object (which is a Hugging Face pipeline for text generation)
# to process the 'history' list. The 'history' list represents the multi-turn conversation
# context, including system, user, and assistant messages. The model will generate a
# response based on this entire conversation history.
# 'max_new_tokens=200' limits the length of the generated response to 200 tokens.
# 'temperature=0.7' controls the creativity of the response, with 0.7 being moderately creative.
# 'do_sample=True' enables sampling, meaning the model will not always pick the most
# probable next token, allowing for more diverse outputs.
multi_output = pipe(history, max_new_tokens=200, temperature=0.7, do_sample=True)
multi = multi_output[0]["generated_text"][-1]["content"]
print("\nMULTI-TURN follow-up:\n", multi)

Technique 3 — Prompt Templates

Concept: A template is a reusable prompt with variable slots. Instead of writing the same prompt structure 50 times, you write it once and fill it in.


Why this matters for your course:
This is exactly how professional AI applications work — they fill templates at runtime based on user input or data. ChatGPT, Claude, Gemini all run on template systems under the hood.


In [ ]:
# Simple Python f-string template (no library needed)
def explain_template(concept, audience, format_style):
    return f"""
Explain the concept of "{concept}" to a {audience}.
Format your response as: {format_style}.
Keep it concise.
"""

# Reuse the same template for three different combinations
print(ask(explain_template("gravity", "5-year-old", "a short story")))
print("---")
print(ask(explain_template("machine learning", "high school student", "3 bullet points")))
print("---")
print(ask(explain_template("the water cycle", "curious adult", "an analogy")))

Technique 4 — Zero-Shot Prompting

Concept: Zero-shot means giving the model a task with NO examples — relying entirely on its training. The quality depends heavily on how clearly you specify the task, role, and format.

In [ ]:
# Bad zero-shot (vague)
bad = ask("Is this positive or negative? The movie was okay I guess.")
print("BAD (vague):\n", bad)

# Good zero-shot (clear role + task + output format)
good = ask("""
You are a sentiment analysis system.
Classify the sentiment of the following text as exactly one of: POSITIVE, NEGATIVE, or NEUTRAL.
Output ONLY the label, nothing else.

Text: "The movie was okay I guess."
""")
print("\nGOOD (clear spec):\n", good)

# Format specification matters
formatted = ask("""
Classify the sentiment of these 3 texts. Return a JSON array only:
1. "I love this product!"
2. "It's fine, not great."
3. "Absolute garbage, wasted my money."

Format: [{"text": "...", "sentiment": "POSITIVE/NEGATIVE/NEUTRAL"}, ...]
""")
print("\nFORMATTED OUTPUT:\n", formatted)

Technique 5 — Few-Shot Learning (In-Context Learning)

Concept: You give the model 2–5 examples of input → output pairs inside the prompt itself. The model learns the pattern from context, not from training. This is called "in-context learning."


Critical insight: The model follows the pattern of our examples, not its own judgment. Garbage examples → garbage outputs. This is a major failure mode in production.

In [ ]:
# Zero-shot baseline (no examples)
zero = ask('Classify: "The sunset was breathtaking." → ')
print("ZERO-SHOT:", zero)

# Few-shot: 3 examples before the actual question
few_shot_prompt = """
Classify each movie review as FRESH or ROTTEN.

Review: "A masterpiece of modern cinema." → FRESH
Review: "I fell asleep twice." → ROTTEN
Review: "Stunning visuals but no plot." → ROTTEN
Review: "The best film I've seen this year." → FRESH

Now classify this:
Review: "The sunset was breathtaking." → """

print("\nFEW-SHOT:", ask(few_shot_prompt, max_new_tokens=10))

# Important: example selection matters
# Try with misleading examples to show sensitivity
misleading_prompt = """
Review: "I loved every second of it." → ROTTEN
Review: "Total disaster of a film." → FRESH
Review: "The acting was world-class." →
"""
print("\nMISLEADING EXAMPLES:", ask(misleading_prompt, max_new_tokens=10))

Technique 6 — Chain of Thought (CoT) Prompting

Concept: Instead of asking for the answer directly, you ask the model to show its reasoning step by step. This dramatically improves accuracy on logic, math, and multi-step problems.

Why it works: Forcing the model to "show its work" catches logical errors mid-generation that would otherwise be buried inside the final answer.

In [ ]:
# Without CoT — direct answer (often wrong on tricky problems)
no_cot = ask("""
A bat and a ball cost $1.10 in total.
The bat costs $1.00 more than the ball.
How much does the ball cost?
""")
print("WITHOUT CoT:\n", no_cot)

# With CoT — "think step by step"
with_cot = ask("""
A bat and a ball cost $1.10 in total.
The bat costs $1.00 more than the ball.
How much does the ball cost?

Think through this step by step before giving your answer.
""")
print("\nWITH CoT:\n", with_cot)

# Explicit step format
explicit_cot = ask("""
A farmer has 17 sheep. All but 9 die. How many are left?

Solve this step by step:
Step 1 - Read the problem carefully and identify the key phrase.
Step 2 - Determine what "all but 9" actually means.
Step 3 - Calculate the answer.
Step 4 - State the final answer clearly.
""")
print("\nEXPLICIT CoT:\n", explicit_cot)

Technique 7 — Self-Consistency

Concept: Generate the same CoT prompt multiple times (with some randomness), collect the different reasoning paths, then take the majority answer. It's like asking 5 people the same question and going with what most of them say.

Note: Self-consistency increases reliability at the cost of speed (5 calls instead of 1). In production, this tradeoff is often worth it for high-stakes outputs.

In [ ]:
import collections

def self_consistent_answer(question, n=5, temperature=0.8):
    """Runs the same CoT prompt n times and returns the majority answer."""
    prompt = f"""
{question}

Think step by step and give your final answer at the very end on a line starting with "ANSWER:".
"""
    answers = []
    for i in range(n):
        response = ask(prompt, temperature=temperature, max_new_tokens=200)
        # Extract the answer line
        for line in response.split("\n"):
            if line.strip().startswith("ANSWER:"):
                answers.append(line.replace("ANSWER:", "").strip())
                break
        print(f"  Run {i+1}: {answers[-1] if answers else '(no ANSWER line found)'}")

    if answers:
        majority = collections.Counter(answers).most_common(1)[0]
        print(f"\n✅ Majority answer ({majority[1]}/{n} votes): {majority[0]}")
    return answers

print("=== SELF-CONSISTENCY TEST ===")
self_consistent_answer("What is 15% of 240?", n=5)

Technique 8 — Tree of Thought (ToT)

Concept: Instead of a single reasoning chain, the model explores multiple branching paths simultaneously and evaluates which branch looks most promising — like a chess player considering several moves ahead.




In [ ]:
# Simple single-prompt ToT simulation
tot_prompt = """
You need to solve this problem: "How would you increase reading rates among teenagers?"

Imagine THREE different experts are each suggesting their best idea.
They each write down one step of their thinking.
Then they discuss and each refine their idea.
Then they collectively decide the strongest approach.

Run this process and show all the steps:

Expert 1 (Educator):
Expert 2 (Tech Designer):
Expert 3 (Teen Psychologist):

[Discussion]

[Agreed strongest approach]
"""

tot_result = ask(tot_prompt, max_new_tokens=500)
print(tot_result)

Technique 9 — Task Decomposition

Concept: Break a complex, multi-part task into smaller subtasks. Feed the output of each subtask as input to the next. This prevents the model from trying to do too much in one generation and losing quality.

In [ ]:
def decomposed_pipeline(topic):
    """Demonstrates task decomposition: each step feeds the next."""

    # Subtask 1: Generate a list of angles
    step1 = ask(f"""
List 4 distinct angles or perspectives from which to discuss "{topic}".
Return only a numbered list, no explanations.
""", max_new_tokens=100)
    print("STEP 1 — Angles:\n", step1)

    # Subtask 2: Pick the most interesting angle
    step2 = ask(f"""
From this list of angles about "{topic}":
{step1}

Which single angle would be most surprising or counterintuitive to a general audience?
Answer in one sentence.
""", max_new_tokens=60)
    print("\nSTEP 2 — Best angle:\n", step2)

    # Subtask 3: Write the actual content
    step3 = ask(f"""
Write a compelling 3-sentence opening paragraph about "{topic}" from this angle:
{step2}

Make it vivid and hook the reader immediately.
""", max_new_tokens=200)
    print("\nSTEP 3 — Final paragraph:\n", step3)

    return step3

decomposed_pipeline("the invention of the printing press")

Technique 10 — Prompt Chaining

Concept: Similar to decomposition, but the chain is designed for dynamic, conditional flow — the output of one prompt determines which prompt fires next. This is the backbone of most production AI systems.

In [ ]:
def prompt_chain(user_input):
    """A simple 3-link chain: classify → route → respond."""

    # Link 1: Classify intent
    intent = ask(f"""
Classify the user's intent as exactly one of: QUESTION, COMPLAINT, COMPLIMENT.
Output only the label.

User: "{user_input}"
""", max_new_tokens=5, temperature=0.0, do_sample=False)
    intent = intent.strip()
    print(f"🔗 Chain link 1 — Intent: {intent}")

    # Link 2: Route to the right response template
    if "QUESTION" in intent:
        prompt = f'Answer this question helpfully and concisely: "{user_input}"'
    elif "COMPLAINT" in intent:
        prompt = f'Respond to this complaint with empathy and offer a solution: "{user_input}"'
    else:
        prompt = f'Respond warmly to this compliment and build on the positivity: "{user_input}"'

    print(f"🔗 Chain link 2 — Routed to: {intent} handler")

    # Link 3: Generate final response
    response = ask(prompt, max_new_tokens=150)
    print(f"🔗 Chain link 3 — Final response:\n{response}")
    return response

prompt_chain("Why does my phone battery drain so fast?")
print("\n---\n")
prompt_chain("This app crashes every single time I open it, completely useless!")

Technique 11 — Crafting Clear Instructions

Concept: The precision and structure of your instructions determine output quality more than any other variable. This technique focuses on four levers: specificity, constraints, output format, and tone.

In [ ]:
# The 4 levers side-by-side
topic = "climate change solutions"

# Lever 1: Vague vs Specific
vague   = ask(f"Write about {topic}.")
specific = ask(f"List exactly 3 {topic} that could be implemented at the city level by 2030. Use bullet points.")
print("VAGUE:\n", vague[:300])
print("\nSPECIFIC:\n", specific)

# Lever 2: No constraint vs Hard constraints
no_constraint = ask(f"Summarize {topic}.")
constrained   = ask(f"Summarize {topic} in exactly 2 sentences. Do not use the word 'important'.")
print("\nNO CONSTRAINT:\n", no_constraint[:200])
print("\nCONSTRAINED:\n", constrained)

# Lever 3: Format matters
markdown_fmt = ask(f"""
Describe 3 {topic}.
Format each as:
### [Solution Name]
**How it works:** [1 sentence]
**Biggest challenge:** [1 sentence]
""")
print("\nMARKDOWN FORMAT:\n", markdown_fmt)

Technique 12 — A/B Testing & Iterative Refinement

Concept: Treat prompts like hypotheses. Run two versions (A/B), compare outputs on a rubric, refine the winner, repeat. This is how production prompt engineering actually works.

In [ ]:
def ab_test(task, prompt_a, prompt_b, rubric):
    """Runs an A/B test on two prompts and asks the model to judge."""
    output_a = ask(prompt_a, max_new_tokens=200)
    output_b = ask(prompt_b, max_new_tokens=200)

    judge_prompt = f"""
You are an impartial evaluator.

TASK: {task}
RUBRIC: {rubric}

RESPONSE A:
{output_a}

RESPONSE B:
{output_b}

Evaluate both responses against the rubric.
Give each a score from 1–10.
State which is better and explain why in 2 sentences.
"""
    verdict = ask(judge_prompt, max_new_tokens=200, temperature=0.3)

    print("--- PROMPT A OUTPUT ---\n", output_a)
    print("\n--- PROMPT B OUTPUT ---\n", output_b)
    print("\n--- JUDGE VERDICT ---\n", verdict)

ab_test(
    task="Explain recursion to a beginner",
    prompt_a="Explain recursion.",
    prompt_b="Explain recursion to someone who has never programmed before. Use a real-world analogy. Keep it under 4 sentences.",
    rubric="Clarity for a complete beginner, use of analogy, conciseness"
)

Technique 13 — Role-Based Prompting (Persona Assignment)

Concept: Giving the model a specific persona changes not just tone but the knowledge frame it draws from. A doctor, a poet, and an engineer will describe the same thing completely differently.

In [ ]:
topic = "the importance of sleep"

roles = [
    ("a neuroscientist", "be precise and cite mechanisms"),
    ("a basketball coach", "relate everything to athletic performance"),
    ("a comedian", "make it funny but still accurate"),
]

for role, instruction in roles:
    response = ask(
        f"Explain {topic} in 3 sentences.",
        system=f"You are {role}. {instruction}."
    )
    print(f"=== AS {role.upper()} ===\n{response}\n")

Technique 14 (18) — Meta-Prompting (Prompts That Write Prompts)

Concept: Ask the model to generate a better prompt than you could write yourself. Then use that generated prompt. This is recursive — and powerful.

In [ ]:
# Step 1: Ask the model to write a better prompt
meta = ask("""
I want to get the best possible explanation of "how neural networks learn"
for a 13-year-old student with no math background.

Write the ideal prompt I should use to get that explanation.
The prompt should include: audience, constraints, format, and a creative framing.
Output ONLY the prompt text itself, nothing else.
""")
print("=== GENERATED PROMPT ===\n", meta)

# Step 2: Now USE the generated prompt
print("\n=== RESPONSE TO GENERATED PROMPT ===")
print(ask(meta, max_new_tokens=400))

Technique 15 (20) — Content Filtering in Prompts

Concept: You can build a lightweight content moderation layer using prompt engineering alone — no separate API needed.

In [ ]:
def content_filter(user_input, category="student aged 13"):
    """Screen user input before passing to main model."""
    verdict = ask(f"""
You are a content moderation system.
The content policy: appropriate for {category}.

Classify this input as SAFE or UNSAFE.
If UNSAFE, explain which policy it violates in one sentence.
Output format: "SAFE" or "UNSAFE: [reason]"

Input: "{user_input}"
""", temperature=0.0, do_sample=False, max_new_tokens=30)
    return verdict.strip()

def safe_ask(user_input, audience="student aged 13"):
    verdict = content_filter(user_input, audience)
    if verdict.startswith("SAFE"):
        return ask(user_input)
    else:
        return f"⚠️ Content blocked — {verdict}"

# Tests
print(safe_ask("What causes earthquakes?"))
print(safe_ask("How do I make explosives?"))
print(safe_ask("Write a poem about autumn leaves."))

Technique 16 (22) — Advanced Composition: Putting It All Together

Concept: A production-grade prompt combines multiple techniques simultaneously — role, format, constraints, chain-of-thought, safety, and output structure. This is your capstone.

In [21]:
def educational_assistant(topic, student_age, learning_goal):
    """
    A complete prompt that combines:
    - Role assignment (Technique 13)
    - Chain of thought (Technique 6)
    - Output formatting (Technique 14)
    - Constraints (Technique 15)
    - Task decomposition (Technique 9)
    """
    system = f"""
You are an expert curriculum designer and educator who specializes in teaching
STEM concepts to students aged {student_age}.

Your communication style:
- Uses relatable analogies from everyday life
- Builds from simple → complex
- Never assumes prior knowledge
- Ends with a question that sparks curiosity
"""

    prompt = f"""
Create a mini-lesson on: "{topic}"
Designed for: {student_age}-year-old students
Learning goal: {learning_goal}

Work through this step by step:

STEP 1 — Hook (1 sentence): Start with a surprising or relatable fact.
STEP 2 — Core concept: Explain the concept using an analogy to something students know.
STEP 3 — How it works: Give a simple 3-step breakdown.
STEP 4 — Real-world application: Where do they encounter this in real life?
STEP 5 — Curiosity question: End with one open question that makes them want to know more.

Constraints:
- Total length: under 250 words
- No jargon without an immediate plain-English definition
- At least one concrete number or measurement
"""

    return ask(prompt, system=system, max_new_tokens=400)

print(educational_assistant(
    topic="how AI models learn from data",
    student_age=13,
    learning_goal="understand that AI learns from patterns, not rules"
))

[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Lesson Plan: How AI models learn from data

Title: "How AI Learns from Data: A Minimalist Approach"

Objective: Students will learn how AI models learn from data using simple but engaging language and examples, and end with a question that sparks curiosity.

Materials:
- Whiteboard or projector
- Handouts with basic math and data concepts
- Interactive whiteboard or screen for demonstrations and visual aids
- Scratch coding software and materials for students to create their own AI models

Standards:
- CCSS.ELA-LITERACY.W.3.2 (Students will analyze how AI models learn from data and how that process is related to the mathematical concepts they are learning.)
- CCSS.ELA-LITERACY.W.4.1 (Students will analyze how AI models learn from data and how that process is related to the mathematical concepts they are learning.)

Introduction:
- Start the lesson by showing a video or demonstration of an AI model, explaining the concept of learning from data and displaying the math or data concepts us